# Connect and Load Data to MySQL

In [ ]:
# Import Rquired Libraries

import pandas as pd
from sqlalchemy import create_engine
import os
import logging
import time

# Create Logging Module

logging.basicConfig(
    filename ="logs/ingestion_db.log",
    level = logging.DEBUG,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a"

)

# Create Connection With MySQL

user = "root"
password = "sanket1234"
host = "localhost"
port = "3306"
database =  "delivery_performance"


try:
    engine = create_engine(
        f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
    )
except:
    print("Connection Error")

def ingestion_db(df, tablename, engine):
     df.to_sql(tablename, con = engine , if_exists  = 'replace', index = False)

def load_row_data():

    start = time.time()
    for file in os.listdir('data_files'):
        if '.csv' in file:
            print(file)


            df = pd.read_csv('data_files/'+file)
            logging.info(f"ingesting {file} in database")
            ingestion_db(df, file[:-4], engine)
    end = time.time()

    total_time = (end - start) / 60
    logging.info("ingestion complete")
    logging.info(f"\n Total Time Taken: {total_time} minutes")

if __name__  == '__main__':
    load_row_data()





geolocation.csv
orders.csv
order_items.csv
payments.csv
products.csv
sellers.csv


# Investigate and Prepare Data

 This Section We Check For Relavent Data From Thses Tables and Create a New Table Which Contain Only Related Data,
After Do The "Data Cleaning" Like Remove Null Values , Duplicates, Correct Data Type ect

In [8]:
tables = pd.read_sql_query("show tables from delivery_performance", con = engine)
tables

,Tables_in_delivery_performance
0,geolocation
1,order_items
2,orders
3,payments
4,products
5,sellers


In [11]:
for table in tables['Tables_in_delivery_performance']:
    print('-' * 70 , f'{table}', '-' * 70)
    print('Count of Records : ', pd.read_sql_query(f"SELECT COUNT(*) AS Count FROM {table}", con = engine)['Count'].values[0])
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", con = engine))


---------------------------------------------------------------------- geolocation ----------------------------------------------------------------------
Count of Records :  1000163


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


---------------------------------------------------------------------- order_items ----------------------------------------------------------------------
Count of Records :  112650


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


---------------------------------------------------------------------- orders ----------------------------------------------------------------------
Count of Records :  99441


...

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


---------------------------------------------------------------------- products ----------------------------------------------------------------------
Count of Records :  32951


,product_id,product category,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,Art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,sport leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,babies,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0


---------------------------------------------------------------------- sellers ----------------------------------------------------------------------
Count of Records :  3095


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


the number of customers comes to order_items are 112650 of with unique are 98666.
the number od customer comes to order something are 99441 of which unique are 99441.
the number of customer done payments are 103886 of which unique are 99440

In [ ]:
pd.read_sql_query("""select count(distinct(order_id)) as order_id from orders where order_status in('delivered', 'invoiced', 'shipped', 'approved', 'created',  'canceled', 'unavailable', 'processing')  """, con = engine)

,order_id
0,99127


In [76]:
pd.read_sql_query("""select distinct(order_status) as se_id from orders""", con = engine)

,se_id
0,delivered
1,invoiced
2,shipped
3,processing
4,unavailable
5,canceled
6,created
7,approved


In [ ]:
pd.read_sql_query("""select count(distinct(order_id)) as se_id from orders """, con = engine)

,se_id
0,99441


In [73]:
pd.read_sql_query("""select count(distinct(order_id)) as se_id from order_items """, con = engine)

,se_id
0,98666


In [72]:
pd.read_sql_query("""select count(distinct(seller_id)) as se_id from sellers """, con = engine)

,se_id
0,3095


In [71]:
pd.read_sql_query("""select count(distinct(seller_id)) as se_id from order_items """, con = engine)

,se_id
0,3095


In [14]:
pd.read_sql_query("""select * from order_items where order_id = ("00010242fe8c5a6d1ba2dd792cb16214")""", con = engine )

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29


In [16]:
pd.read_sql_query("""select * from payments where order_id = ('00010242fe8c5a6d1ba2dd792cb16214')""", con = engine)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,credit_card,2,72.19


In [25]:
pd.read_sql_query("""select * from order_items where order_item_id = ('1')""", con = engine )

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
98662,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
98663,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
98664,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [60]:
pd.read_sql_query("""select order_id  from order_items where order_item_id = ('19')""", con = engine)

,order_id
0,1b15974a0141d54e36626dca3fdc731a
1,8272b63d03f5f79c56e9e4120aec44ef
2,ab14fdcfbe524636d65ee38360e22ce8


In [28]:
pd.read_sql_query("""select * from payments where order_id = ('00010242fe8c5a6d1ba2dd792cb16214')""", con = engine)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,credit_card,2,72.19


In [34]:
pd.read_sql_query("""select  order_id, sum(price) as total_price, sum(freight_value) as total_freight_value from order_items group by order_id""", con = engine)

,order_id,total_price,total_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14
...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,299.99,43.41
98662,fffcd46ef2263f404302a634eb57f7eb,350.00,36.53
98663,fffce4705a9662cd70adb13d4a31832d,99.90,16.95
98664,fffe18544ffabc95dfada21779c9644f,55.99,8.72


In [ ]:
# Combine 5 Tables orders. order_items, payments, products, sellers

pd.read_sql_query("""
select 
	ord.order_id, 
        ord.order_item_id, 
        ord.product_id, 
        ord.seller_id, 
        ord.shipping_limit_date, 
        ord.price, 
        ord.freight_value,
        py.payment_sequential,
        py.payment_type,
        py.payment_installments,
        py.payment_value,
        pd.product_category,
        pd.product_weight_g,
        pd.product_length_cm,
        pd.product_height_cm,
        pd.product_width_cm,
        se.seller_city,
        se.seller_state,
        se.seller_zip_code_prefix,
        ords.customer_id,
        ords.order_status,
        ords.order_purchase_timestamp,
        ords.order_approved_at,
        ords.order_delivered_carrier_date,
        ords.order_delivered_customer_date,
        ords.order_estimated_delivery_date
        
	
        
from order_items as ord join payments as py
on ord.order_id = py.order_id
join products as pd
on ord.product_id = pd.product_id
join sellers as se
on ord.seller_id = se.seller_id
join orders as ords
on ord.order_id = ords.order_id
        """, con = engine)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,...,seller_city,seller_state,seller_zip_code_prefix,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,02951078a21a2d9341ea16089a4d5031,1,dd3575a8c5e2139f680a9816a15c8f2a,4869f7a5dfa277a7dca6462dcf3b52b2,2017-03-31 16:10:18,235.00,22.10,1,credit_card,12,...,guariba,SP,14840,6c51ed8c29ff8bc5f3b649dc7315b787,delivered,2017-03-16 15:16:53,2017-03-23 16:10:18,2017-03-24 14:42:45,2017-04-05 17:04:39,2017-04-19 00:00:00
1,0132158c30672a8c52997a2492613ec2,1,e9eebb8e8ba0fadb9020f8ba1c003b48,620c87c171fb2a6dd6e8bb4dec959fc6,2017-03-19 23:23:23,399.90,13.41,1,credit_card,8,...,petropolis,RJ,25645,5a3f73cc3e52ab64478beb5ce8756f4b,delivered,2017-03-13 23:23:23,2017-03-13 23:23:23,2017-03-14 09:35:36,2017-03-15 09:27:54,2017-03-30 00:00:00
2,023b49300b34f2b9961857266df731dc,1,d918b3f4aa5272c2c3cd088d087ca069,c3cfdc648177fdbbbb35635a37472c53,2017-09-29 10:10:20,139.90,22.85,1,credit_card,2,...,curitiba,PR,80610,0d0ce626603ca795728958e76c79785e,delivered,2017-09-21 09:58:40,2017-09-21 10:10:20,2017-09-21 21:40:06,2017-09-28 21:48:49,2017-10-19 00:00:00
3,0159c6355a4e32f6ac68d838e2228150,1,a00722035cea70bbf671b758459cde42,25be943a321c8938947bdaabca979a90,2018-06-21 14:03:58,189.00,53.11,1,credit_card,15,...,sao bernardo do campo,SP,9725,bf0176bae5facd261148fb881ccb8cc6,delivered,2018-06-15 13:41:08,2018-06-15 14:03:58,2018-06-18 12:06:00,2018-06-19 23:21:34,2018-07-03 00:00:00
4,028e8f5a96db8d372890e03d22829e9b,1,d2f5484cbffe4ca766301b21ab9246dd,36a968b544695394e4e9d7572688598f,2018-04-24 17:58:33,12.88,7.87,1,credit_card,1,...,santos,SP,11010,c09962ce82ae80e545818b93530aab09,delivered,2018-04-18 17:15:44,2018-04-18 17:58:33,2018-04-19 23:21:25,2018-04-21 00:11:03,2018-04-30 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117596,321ebafc185998f3acb39280b759b177,1,d0ff65b2782e494c2c5b66e19a6e2902,53e4c6e0f4312d4d2107a8c9cddf45cd,2017-07-04 17:04:02,29.60,12.69,1,credit_card,4,...,pedreira,SP,13920,0b6160d3adbecfd48f0d5ba84e77e913,delivered,2017-06-28 16:45:19,2017-06-28 17:32:21,2017-07-03 15:26:36,2017-07-06 23:33:27,2017-07-25 00:00:00
117597,0a89bc2ec7e822a78b0cef45518d6217,1,4dc44f95ece9249f0e50d10542ca6c03,fa1c13f2614d7b5c4749cbc52fecda94,2017-11-20 23:55:29,399.90,28.08,1,credit_card,8,...,sumare,SP,13170,a80a94cf8bd343cdd10d03d8b37ad719,delivered,2017-11-13 23:44:17,2017-11-13 23:55:29,2017-11-14 18:49:15,2017-11-24 16:21:26,2017-12-13 00:00:00
117598,52b226b1528922785f17fc4dadbe7110,1,cf5407ad3a5c603fa46d5c2613661a09,dd7ddc04e1b6c2c614352b383efe2d36,2018-03-20 14:15:42,29.90,14.44,1,credit_card,1,...,sao paulo,SP,3471,5d223584b58b69a22b86aacdf4d12b01,delivered,2018-03-12 13:50:41,2018-03-12 14:15:42,2018-03-13 20:33:05,2018-03-21 22:57:43,2018-04-02 00:00:00
117599,c85e13fa13301ec2cb996db10353e737,1,cfb763496d9fc48751a27db4fd02aa2d,391fc6631aebcf3004804e51b40bcf1e,2018-07-22 22:45:15,49.95,18.45,1,credit_card,3,...,ibitinga,SP,14940,3435843eaf57bad18bf0d39305d6f5dd,delivered,2018-07-17 22:35:29,2018-07-17 22:45:15,2018-07-19 11:50:00,2018-07-27 18:24:46,2018-08-07 00:00:00


In [ ]:
pd.read_sql("""
    select 
		ifnull(se.seller_id, 'NA') as seller_id,
        ifnull(se.seller_zip_code_prefix, 'NA') as seller_zip_code_prefix,
        ifnull(se.seller_city,'NA') as seller_city,
        ifnull(se.seller_state, 'NA') as seller_state,
        ge.geolocation_zip_code_prefix,
        ge.geolocation_city, 
        ge.geolocation_state
        
	from geolocation as ge
	left join sellers as se
	on ge.geolocation_zip_code_prefix = se.seller_zip_code_prefix""", con = engine)

,seller_id,seller_zip_code_prefix,seller_city,seller_state,geolocation_zip_code_prefix,geolocation_city,geolocation_state
0,NA,NA,NA,NA,1037,sao paulo,SP
1,NA,NA,NA,NA,1046,sao paulo,SP
2,NA,NA,NA,NA,1046,sao paulo,SP
3,e5cbe890e679490127e9a390b46bbd20,1041,sao paulo,SP,1041,sao paulo,SP
4,1d503743d2526f03f0c2c89540ee008c,1035,sao paulo,SP,1035,sao paulo,SP
...,...,...,...,...,...,...,...
1166429,NA,NA,NA,NA,99950,tapejara,RS
1166430,NA,NA,NA,NA,99900,getulio vargas,RS
1166431,NA,NA,NA,NA,99950,tapejara,RS
1166432,NA,NA,NA,NA,99980,david canabarro,RS


In [ ]:
delivery_delay_summary = pd.read_sql("""with customer_order_info as(
    select 
        ord.order_id, 
        ord.order_item_id, 
        ord.product_id, 
        ord.seller_id, 
        ord.shipping_limit_date, 
        ord.price, 
        ord.freight_value,
        py.payment_sequential,
        py.payment_type,
        py.payment_installments,
        py.payment_value,
        pd.product_category,
        pd.product_weight_g,
        pd.product_length_cm,
        pd.product_height_cm,
        pd.product_width_cm,
        se.seller_city,
        se.seller_state,
        se.seller_zip_code_prefix,
        ords.customer_id,
        ords.order_status,
        ords.order_purchase_timestamp,
        ords.order_approved_at,
        ords.order_delivered_carrier_date,
        ords.order_delivered_customer_date,
        ords.order_estimated_delivery_date
        
	
        
    from order_items as ord join payments as py
    on ord.order_id = py.order_id
    join products as pd
    on ord.product_id = pd.product_id
    join sellers as se
    on ord.seller_id = se.seller_id
    join orders as ords
    on ord.order_id = ords.order_id                               
), 
                                    
                                     
location_info as (   
    select 
		ifnull(se.seller_id, 'NA') as seller_id,
        ifnull(se.seller_zip_code_prefix, 'NA') as seller_zip_code_prefix,
        ifnull(se.seller_city,'NA') as seller_city,
        ifnull(se.seller_state, 'NA') as seller_state,
        ge.geolocation_zip_code_prefix,
        ge.geolocation_city, 
        ge.geolocation_state
        
	from geolocation as ge
	left join sellers as se
	on ge.geolocation_zip_code_prefix = se.seller_zip_code_prefix
)
                                     
select  ord.product_id,    
        ord.price, 
        ord.freight_value,
        py.payment_type,
        pd.product_category,
        pd.product_weight_g, 
        se.seller_city,
        se.seller_state,
        se.seller_zip_code_prefix, 
        ords.customer_id,
        ords.order_status,
        ord.shipping_limit_date,
        ords.order_purchase_timestamp,
        ords.order_approved_at,
        ords.order_delivered_carrier_date,
        ords.order_delivered_customer_date,
        ords.order_estimated_delivery_date 
                                     
                                  """)